# 🧑‍⚖️ LLM-as-Judge — Evaluate RAG Automatically

## The Problem

How do you know if your RAG system is giving **good answers**?

Manual evaluation is slow and expensive. Traditional metrics like BLEU are too crude for open-ended answers.

## The Solution — Use an LLM to Judge

Ask a capable LLM (like Claude) to rate the quality of answers on specific dimensions:

```
Faithfulness:     Is the answer supported by the retrieved context?
Relevance:        Does the answer actually address the question?
Completeness:     Does it cover the key points?
```

This is the foundation of **RAGAS** — the most popular RAG evaluation framework.

In [1]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()  # reads ANTHROPIC_API_KEY from .env if present

client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

def llm(prompt, max_tokens=400):
    """Call Claude Haiku. Fast and cheap — perfect for RAG pipelines."""
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text.strip()

print("✅ Anthropic client ready!")
print("   Model: claude-haiku-4-5-20251001")
print("   Set ANTHROPIC_API_KEY in .env or environment.")


✅ Anthropic client ready!
   Model: claude-haiku-4-5-20251001
   Set ANTHROPIC_API_KEY in .env or environment.


In [2]:
# We'll implement LLM-as-Judge scoring manually
# so you understand exactly what RAGAS is doing under the hood

# Simulated RAG outputs to evaluate
examples = [
    {
        "question":  "What causes overfitting?",
        "context":   "Overfitting occurs when a model memorizes training data including noise, leading to poor generalization. Common fixes include dropout, early stopping, and more training data.",
        "answer":    "Overfitting is caused by the model memorizing training data too closely, especially noise. It can be prevented with dropout and early stopping.",
        "label":     "good"
    },
    {
        "question":  "What causes overfitting?",
        "context":   "Overfitting occurs when a model memorizes training data including noise, leading to poor generalization.",
        "answer":    "Overfitting is caused by having too many neurons in the hidden layers of the network.",  # hallucinated detail
        "label":     "hallucinated"
    },
    {
        "question":  "What causes overfitting?",
        "context":   "Overfitting occurs when a model memorizes training data.",
        "answer":    "Machine learning is a subset of artificial intelligence.",  # irrelevant
        "label":     "irrelevant"
    },
]

print(f"{len(examples)} examples to evaluate")

3 examples to evaluate


In [3]:
# The judge prompts — these are sent to the LLM

def faithfulness_prompt(question, context, answer):
    return f"""Rate whether the answer is fully supported by the context.
Score 1-5 where: 1=contradicts context, 3=partially supported, 5=fully supported.
Reply with just a number.

Context: {context}
Answer: {answer}
Score:"""

def relevance_prompt(question, answer):
    return f"""Rate whether the answer addresses the question.
Score 1-5 where: 1=completely irrelevant, 3=partially relevant, 5=directly answers the question.
Reply with just a number.

Question: {question}
Answer: {answer}
Score:"""

print("Faithfulness prompt example:")
print(faithfulness_prompt(
    examples[0]["question"],
    examples[0]["context"],
    examples[0]["answer"]
))

Faithfulness prompt example:
Rate whether the answer is fully supported by the context.
Score 1-5 where: 1=contradicts context, 3=partially supported, 5=fully supported.
Reply with just a number.

Context: Overfitting occurs when a model memorizes training data including noise, leading to poor generalization. Common fixes include dropout, early stopping, and more training data.
Answer: Overfitting is caused by the model memorizing training data too closely, especially noise. It can be prevented with dropout and early stopping.
Score:


In [4]:
# Run the LLM judge on each example

def judge_faithfulness(context, answer):
    """Ask Claude to score faithfulness 1-5."""
    prompt = faithfulness_prompt("", context, answer)
    score_str = llm(prompt, max_tokens=5)
    try:
        return max(1, min(5, int(score_str.strip()[0])))
    except:
        return 3  # default if parse fails

def judge_relevance(question, answer):
    """Ask Claude to score answer relevance 1-5."""
    prompt = relevance_prompt(question, answer)
    score_str = llm(prompt, max_tokens=5)
    try:
        return max(1, min(5, int(score_str.strip()[0])))
    except:
        return 3


import numpy as np
print(f"{'Label':<15} {'Faithfulness':>13} {'Relevance':>11} {'Overall':>9}")
print("-" * 52)

for ex in examples:
    f_score = judge_faithfulness(ex['context'], ex['answer'])
    r_score = judge_relevance(ex['question'], ex['answer'])
    overall = (f_score + r_score) / 2
    emoji   = "✅" if overall >= 4 else ("⚠️ " if overall >= 2.5 else "❌")
    print(f"{ex['label']:<15} {f_score:>13} {r_score:>11} {overall:>7.1f} {emoji}")


Label            Faithfulness   Relevance   Overall
----------------------------------------------------
good                        5           4     4.5 ✅
hallucinated                2           3     2.5 ⚠️ 
irrelevant                  2           1     1.5 ❌


In [6]:
# Batch evaluation — run across your entire test set

def evaluate_rag_system(test_examples):
    """Evaluate a set of RAG outputs. Returns average scores."""
    all_faithfulness = []
    all_relevance    = []

    for ex in test_examples:
        f_score = judge_faithfulness(ex['context'], ex['answer'])
        r_score = judge_relevance(ex['question'], ex['answer'])

        all_faithfulness.append(f_score)
        all_relevance.append(r_score)

    return {
        "faithfulness": np.mean(all_faithfulness),
        "relevance":    np.mean(all_relevance),
        "n_evaluated":  len(test_examples),
    }

import numpy as np
results = evaluate_rag_system(examples)
print("RAG System Evaluation Results")
print("-" * 35)
print(f"  Faithfulness : {results['faithfulness']:.2f} / 5")
print(f"  Relevance    : {results['relevance']:.2f} / 5")
print(f"  Evaluated    : {results['n_evaluated']} examples")
print()
print("💡 Run this after each change to your RAG pipeline to track improvement.")

RAG System Evaluation Results
-----------------------------------
  Faithfulness : 2.67 / 5
  Relevance    : 2.67 / 5
  Evaluated    : 3 examples

💡 Run this after each change to your RAG pipeline to track improvement.


## Key Takeaways

**LLM-as-Judge gives you:**
- Automatic evaluation at scale (no human labelling)
- Nuanced scoring (not just exact match)
- Actionable signals: faithfulness tells you about hallucination, relevance about retrieval quality

**Ready-made frameworks that use this pattern:**
```python
# RAGAS — most popular RAG evaluation library
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy

result = evaluate(dataset, metrics=[faithfulness, answer_relevancy])
print(result)  # {"faithfulness": 0.91, "answer_relevancy": 0.87}
```

**Paper:** Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena (Zheng et al., 2023)